# Silver: CRM sales
**Source:** `bronze.crm_sales_details` -> **Target:** `silver.crm_sales`

**What this notebook does:**
- Remove extra spaces
- Turn number dates (20101229) into real dates; bad ones become empty
- **Fix price** (missing or negative)
- **Fix sales amount** (must equal quantity × price)
- Rename columns
- **Quarantine bad rows**: sales with no matching customer or product (or an amount that cannot be fixed) go to `silver.crm_sales_rejects` with a reason, instead of flowing into Gold

**Run order:** this notebook needs `silver.crm_customers` and `silver.crm_products`, so run it **after** those two.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

CATALOG = "workspace"

## Read the Bronze table

In [0]:
df = spark.table(f"{CATALOG}.bronze.crm_sales_details")

## 1. Trim spaces

In [0]:
# Remove extra spaces from every text column
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## 2. Clean the dates
Dates come as numbers like `20101229`. A valid one has 8 digits. Anything else (like `0`) becomes empty.

In [0]:
def clean_date(column_name):
    """Turn 20101229 into a date. Return empty (null) if the value is not a valid 8-digit date."""
    as_text = F.col(column_name).cast("string")
    return F.when(F.length(as_text) == 8, F.to_date(as_text, "yyyyMMdd"))

df = (
    df
    .withColumn("sls_order_dt", clean_date("sls_order_dt"))
    .withColumn("sls_ship_dt",  clean_date("sls_ship_dt"))
    .withColumn("sls_due_dt",   clean_date("sls_due_dt"))
)

## 3. Fix the price
If price is missing or not positive, calculate it: `sales / quantity`.

In [0]:
price_is_bad = F.col("sls_price").isNull() | (F.col("sls_price") <= 0)

df = df.withColumn(
    "sls_price",
    F.when(price_is_bad & (F.col("sls_quantity") != 0),
           F.col("sls_sales") / F.col("sls_quantity"))
     .otherwise(F.col("sls_price"))
)

## 4. Fix the sales amount
Rule: `sales = quantity × price`. If sales is missing, not positive, or does not match, recalculate it.

In [0]:
expected_sales = F.col("sls_quantity") * F.abs(F.col("sls_price"))

sales_is_bad = (
    F.col("sls_sales").isNull()
    | (F.col("sls_sales") <= 0)
    | (F.col("sls_sales") != expected_sales)
)

df = df.withColumn(
    "sls_sales",
    F.when(sales_is_bad, expected_sales).otherwise(F.col("sls_sales"))
)

## 5. Rename columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## 6. Quarantine bad rows
A sale is only useful if we know **who** bought and **what** was bought. We check every sale against the Silver customers and the current Silver products.

Rows that fail go to a separate **rejects table** with the reason. This way nothing disappears silently, and bad rows never reach Gold.

In [0]:
customers = (
    spark.table(f"{CATALOG}.silver.crm_customers")
         .select("customer_id").distinct()
         .withColumn("customer_found", F.lit(True))
)
products = (
    spark.table(f"{CATALOG}.silver.crm_products")
         .filter("end_date IS NULL")                 # current version of each product
         .select("product_number").distinct()
         .withColumn("product_found", F.lit(True))
)

df = (
    df.join(customers, "customer_id",    "left")
      .join(products,  "product_number", "left")
)

# Collect every reason a row can fail (a row can have more than one)
reasons = F.array(
    F.when(F.col("customer_found").isNull(), F.lit("customer_not_found")),
    F.when(F.col("product_found").isNull(),  F.lit("product_not_found")),
    F.when(F.col("sales_amount").isNull() | F.col("price").isNull(), F.lit("amount_cannot_be_fixed")),
)
df = (
    df.withColumn("reject_reason", F.concat_ws(", ", reasons))
      .select(                                   # fixed, readable column order
          "order_number", "product_number", "customer_id",
          "order_date", "ship_date", "due_date",
          "sales_amount", "quantity", "price", "reject_reason")
)

valid_rows   = df.filter(F.col("reject_reason") == "").drop("reject_reason")
rejected_rows = (
    df.filter(F.col("reject_reason") != "")
      .withColumn("rejected_at", F.current_timestamp())
)

## Write the Silver tables
- `crm_sales`: good rows only (Gold reads this)
- `crm_sales_rejects`: bad rows with the reason (for you to review)

In [0]:
valid_rows.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.crm_sales")
rejected_rows.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.crm_sales_rejects")

## Check it Quickly

In [0]:
good = spark.table(f"{CATALOG}.silver.crm_sales")
bad  = spark.table(f"{CATALOG}.silver.crm_sales_rejects")
print("good rows:", good.count(), "| rejected rows:", bad.count())
bad.groupBy("reject_reason").count().display()